### Load the embedding model
Loads `all-MiniLM-L6-v2`, a small sentence-transformer that turns text into 384-dimensional vectors.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

### Helper functions
- `cosine_similarity`: how similar two vectors are (1 = same direction).
- `chunk_text`: splits text into 150-word chunks that overlap by 30 words.
- `retrieve`: embeds the query and returns the `k` most similar chunks.
- `expand_with_neighbors`: adds the chunks on either side of each result.
- `find_chunk`: returns the IDs of chunks that contain a keyword (useful for finding gold chunks).

In [ ]:
def cosine_similarity(a, b):
    return (np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
 

def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    result = []
    for start in range(0, len(words), chunk_size - overlap):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        result.append({"chunk_id": len(result), "text": chunk})
        if end >= len(words):
            break
    return result

def retrieve(query, k=3):
    query_vec = model.encode(query)
    scores = []
    for ch_v, ch in zip(chunk_vector, chunks):
        sim = float(cosine_similarity(ch_v, query_vec))
        scores.append({"id": ch["chunk_id"], "score": sim, "chunk": ch})
        
    s_scores = sorted(scores, key=lambda x: x["score"], reverse=True)
    return s_scores[:k]

def expand_with_neighbors(results, window=1):
    ids = set()
    for r in results:
        center = r["id"]
        for neighbor in range(center - window, center + window + 1):
            if 0 <= neighbor < len(chunks):
                ids.add(neighbor)
    return [chunks[i] for i in sorted(ids)]

def find_chunk(keyword):
    return [c["chunk_id"] for c in chunks if keyword.lower() in c["text"].lower()]

### Load the document
Reads `doc.txt` (the gym and health handbook) into `text`.

In [ ]:
with open("doc.txt", encoding= "utf-8") as f:
    text = f.read()

### Chunk the document
Splits the text into overlapping chunks and prints how many there are.

In [ ]:
chunks = chunk_text(text)
print(f"{len(chunks)} Chunks")

### Embed the chunks
Encodes every chunk once and stores the vectors in `chunk_vector`, which `retrieve` searches.

In [ ]:
chunk_vector = model.encode([c["text"] for c in chunks])

### Keyword lookup
Example of `find_chunk`: lists the chunks that mention "protein".

In [ ]:
find_chunk("protein")

### Evaluation set
Ten test questions, each with its gold chunk IDs: the chunks that contain the answer.

In [ ]:
eval_set = [
    {"query": "how much protein should I eat per day?",                 "gold": [27, 28]},
    {"query": "can I bring a friend with me to the gym?",               "gold": [48]},

    {"query": "why are my legs still sore two days after squats?",      "gold": [36, 37]},
    {"query": "my weight isn't changing, how do I know if I'm losing fat?", "gold": [44]},
    {"query": "how many days off should a beginner take each week?",    "gold": [36]},

    {"query": "can I use my student discount on the Titan plan?",       "gold": [47]},
    {"query": "is it fine to do HIIT every day?",                       "gold": [24]},
    {"query": "should I hold long stretches before lifting heavy?",     "gold": [21]},

    {"query": "what does error IH-409 mean?",                           "gold": [51]},

    {"query": "is creatine bad for my kidneys?",                        "gold": [32]},
]

### Recall@k
For each question, checks whether any gold chunk appears in the top `k` results. Prints the misses and the overall recall score.

In [ ]:
def recall_at_k(eval_set, k=3):    
    count = 0
    for q in eval_set:
        retrieved_ids = [r["id"] for r in retrieve(q["query"], k=k)]
        if set(q["gold"]) & set(retrieved_ids):
            count += 1
        else:
            print(f"MISS: {q['query']}  gold={q['gold']}  got={retrieved_ids}")

    print(f"recall@{k} = {count / len(eval_set):.2f}")

recall_at_k(eval_set, k=3)

### Interactive search
Type a question to see the top 3 chunks with their similarity scores. Enter `q` to quit.

In [ ]:
def interactive_search():
    while True:
        query = input("\nQuestion (or 'q' to quit): ")
        if query == 'q':
            break
        for r in retrieve(query, k=3):
            print(f"\n[{r['id']}] {r['score']:.3f}\n{r['chunk']['text'][:300]}...")

### RAG answer function
`answer(query)` retrieves the top 3 chunks, puts them into a prompt with strict "use only the context" instructions, and asks Gemini for an answer. If the context doesn't contain the answer, the model replies "I couldn't find that in the documents." Needs `GEMINI_API_KEY` in `.env`.

In [ ]:
from dotenv import load_dotenv
from google import genai

system_instructions = """Answer the question using ONLY the context below.
If the context does not contain the answer, reply exactly: "I couldn't find that in the documents."
Do not use outside knowledge, even if you think the context is wrong."""
def answer(query):
    results = retrieve(query, k=3)
    context = "\n\n".join(f"[chunk {r['id']}]\n{r['chunk']['text']}" for r in results)
    prompt = f"{system_instructions}\n\n<context>\n{context}\n</context>\n\nQuestion: {query}"
    
    load_dotenv()
    client = genai.Client()
    interaction = client.interactions.create(
        model = "gemini-3.5-flash",
        input = prompt
    )
    
    return interaction.output_text

### Try a question
Runs one question through the full RAG pipeline.

In [ ]:
answer("can I bring a friend with me to the gym?")

### Answer evaluation (`answer(query)`, k=3)

1. How much protein should I eat per day? — **correct**
2. Can I bring a friend with me to the gym? — **false refusal**
3. Why are my legs still sore two days after squats? — **correct**
4. My weight isn't changing, how do I know if I'm losing fat? — **correct**
5. How many days off should a beginner take each week? — **false refusal**
6. Can I use my student discount on the Titan plan? — **correct**
7. Is it fine to do HIIT every day? — **correct**
8. Should I hold long stretches before lifting heavy? — **correct**
9. What does error IH-409 mean? — **correct**
10. Is creatine bad for my kidneys? — **correct**

**Result:** 8 correct, 2 false refusals, 0 wrong